# 07.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/04-arima-sarima/solutions.ipynb)

In [1]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter("ignore", ConvergenceWarning)   # optimiser notices on short series; we check results ourselves
warnings.filterwarnings("ignore", message="Non-")      # "Non-stationary/Non-invertible starting parameters"
warnings.filterwarnings("ignore", category=FutureWarning, message="adfuller")   # upcoming API change notice

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

def sarima(train, h):
    """SARIMA(1,0,1)(0,1,0,52) with a constant: last year's week, corrected by recent errors."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # statsmodels start-value notices, not errors
        model = SARIMAX(train, order=(1, 0, 1), seasonal_order=(0, 1, 0, 52), trend="c").fit(disp=False)
    return model.forecast(h).values

768,341 order lines, 2009-12-01 to 2011-12-04
734 days, 105 weeks, 24 months


## Exercise 1

(a) Differences: 10, −5, 15, 10.

(b) 30 + 0.7 × 60 = 72 → 30 + 0.7 × 72 = 80.4 → 30 + 0.7 × 80.4 = 86.28. Long-run mean = 30 / (1 − 0.7) = **100**.

In [2]:
print(pd.Series([200, 210, 205, 220, 230]).diff().dropna().tolist())
y_ = 60
for _ in range(3):
    y_ = 30 + 0.7 * y_; print(round(y_, 2))
print("long-run mean:", round(30 / (1 - 0.7), 2))

[10.0, -5.0, 15.0, 10.0]
72.0
80.4
86.28
long-run mean: 100.0


## Exercise 2

In [3]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def make_sarima(order):
    def fn(train, h):
        return SARIMAX(train, order=order, seasonal_order=(0, 1, 0, 52), trend="c").fit(disp=False).forecast(h).values
    return fn

res = {str(o): backtest(make_sarima(o))["MAE"].mean() for o in [(1, 0, 1), (1, 0, 0), (0, 0, 1), (0, 0, 0)]}
res["seasonal naive"] = backtest(seasonal_naive)["MAE"].mean()
(pd.Series(res).sort_values() / 1000).round(1).rename("mean MAE £k")

seasonal naive    32.4
(1, 0, 1)         34.6
(0, 0, 0)         34.9
(0, 0, 1)         35.0
(1, 0, 0)         36.1
Name: mean MAE £k, dtype: float64

SARIMA(0,0,0)(0,1,0,52) with a constant is **seasonal naive plus a fixed weekly adjustment**: "same week last year, plus the average year-on-year change". With no AR or MA terms to chase recent errors, it is the most stable of the four. None of the variants beats plain seasonal naive by much, if at all; the extra terms mostly fit noise in 46 weeks of differences.

## Exercise 3

In [4]:
inside, total = 0, 0
for o in ORIGINS:
    tr = weekly[weekly.index <= o]; te = weekly[weekly.index > o].iloc[:H]
    ci = SARIMAX(tr, order=(1, 0, 1), seasonal_order=(0, 1, 0, 52), trend="c").fit(disp=False) \
            .get_forecast(len(te)).conf_int()
    inside += ((te.values >= ci.iloc[:, 0].values) & (te.values <= ci.iloc[:, 1].values)).sum()
    total += len(te)
print(f"{inside} of {total} actual weeks inside the 95% interval = {inside / total:.0%}")

34 of 40 actual weeks inside the 95% interval = 85%


Only **85%** of actual weeks (34 of 40) fell inside a "95%" interval: the intervals are too narrow. The model is over-confident, which is common when the history is short and calendar effects such as bank holidays are not modelled; the misses cluster around the bank-holiday weeks and the September ramp-up. For stock planning, widen the range (for example, plan best-sellers to the upper bound of a 99% interval) or calibrate the width using backtest errors directly.

## Exercise 4

In [5]:
mtrain = monthly[monthly.index <= "2011-10-31"]
f = SARIMAX(mtrain, order=(0, 0, 0), seasonal_order=(0, 1, 0, 12), trend="c").fit(disp=False)
pred = f.forecast(1).iloc[0]; actual = monthly.loc["2011-11-30"]
print(f"November 2011: forecast £{pred:,.0f}, actual £{actual:,.0f}, error {100 * (pred - actual) / actual:+.1f}%")

November 2011: forecast £1,149,688, actual £1,156,206, error -0.6%


With only 23 training months, the seasonal-difference model is "November 2010 plus the average year-on-year change". It lands within **0.6%** of the actual November, closer than the monthly Holt-Winters from 07.3 (+5.6%). Do not read too much into it: this is one forecast of one month, and with a single previous November neither model can claim to be reliably better.